# pyvoa — illustrative examples of the software paper

**pyvoa: a Python framework for unified access, standardisation and visualisation
of open epidemiological data**
T. Beau, J. Browaeys, O. Dadoun — submitted to *SoftwareX*

This notebook runs every code listing of the manuscript, in the order in which
they appear, and produces the manuscript figures. Each code cell below is
**identical to the listing printed in the paper**, so that what is published and
what is executed can be compared line by line rather than trusted.

Two cells are not manuscript listings and are labelled as such: the standalone
use of the geolocation layer (§2.1 and §4), and the wastewater / reported
incidence comparison that illustrates §4, point 1.

Run order matters only in that `setwhom()` selects the active database; each
example sets its own, so cells can also be run individually.

| | |
|---|---|
| repository | <https://github.com/pyvoa/pyvoa> |
| documentation | <https://pyvoa.org> |
| archived release | <https://doi.org/10.5281/zenodo.21829901> |
| licence | MIT |

## 0. Setup

`pyvoa` alone returns data; `pyvoa-full` adds the plotting backends, which the
figures below need. On Google Colab or Binder, run the install cell once; on a
local machine where the package is already installed, skip it.

The first `setwhom()` of a session downloads the dataset into
`~/.cache/pyvoa.data_<user>/` and takes a few minutes. Later runs read the cache,
and `setwhom(db, reload=False)` reuses it without checking upstream at all.

In [ ]:
# Run once on Colab / Binder, or on a fresh environment.
# %pip install pyvoa-full

In [ ]:
import pyvoa.front as pf
import pyvoa.tools as pt

pt.set_verbose_mode(1)          # 0 silent, 1 info, 2 debug
print('pyvoa', pf.getversion())

### Figure directory, and the values the check cell reuses

The listing cells below spell out their arguments literally, so that each is
character-for-character the listing printed in the paper. The two values that
also appear in the check cell of §1 are named here as well, and must be kept in
step with the listings if either changes.

In [ ]:
JHU_DATE  = '01/12/2022'   # JHU CSSE stopped collecting in March 2023
OWID_VACC = 'total_people_vaccinated_per_hundred'
TILE      = 'positron'     # not the default 'openstreet': see the map cells

from pathlib import Path

# The figures belong next to the manuscript that includes them, which is where
# paper_examples.py writes them too. Falls back to ./figures when this notebook
# is run outside a checkout (Colab, a downloaded copy).
FIGDIR = Path('../../paper/figures')
if not FIGDIR.parent.exists():
    FIGDIR = Path('figures')
FIGDIR.mkdir(parents=True, exist_ok=True)

def save(name):
    '''Write the current figure into FIGDIR. Never abort a run on failure:
    savefig support differs between backends.'''
    try:
        pf.savefig(str(FIGDIR / name))
        print('->', FIGDIR / name)
    except Exception as exc:
        print('!! savefig failed for', name, ':', exc)

## 1. Vocabulary check *(recommended before submission)*

This cell asserts that every database, indicator, option and grouping cited in
the manuscript exists in the installed version. It guards against the class of
error that made the first draft's listings unrunnable: a database that was never
in the catalogue, an option renamed in 0.4.0, an indicator absent from the
source's JSON descriptor.

Run it again after every version bump.

In [ ]:
failures = []

def want(label, value, allowed):
    if value in allowed:
        print(f'  ok   {label}: {value!r}')
    else:
        failures.append(f'{label}: {value!r} not in {sorted(allowed)}')
        print(f'  FAIL {label}: {value!r}')

whom = set(pf.listwhom())
for db in ('owid', 'jhu', 'spf', 'spfnational', 'sumeau',
           'measles-usa', 'ebolardc'):
    want('database', db, whom)

want('what',       'daily',   set(pf.listwhat()))
want('what',       'current', set(pf.listwhat()))
want('option',     'smooth7', set(pf.listoption()))
want('output',     'pandas',  set(pf.listoutput()))

# listhist(), listplot() and listmap() read the chart vocabulary of the selected
# backend, so they raise until setvis() has been called.
pf.setvis('matplotlib')
want('typeofhist', 'location', set(pf.listhist()))
want('typeofplot', 'yearly',   set(pf.listplot()))
want('typeofmap',  'dense',    {str(m) for m in pf.listmap()})

print('\nindicators (needs setwhom, hence a download):')
for db, indicators in (('owid',        ('total_deaths', OWID_VACC)),
                       ('jhu',         ('tot_confirmed',)),
                       ('spf',         ('cur_hosp',)),
                       ('spfnational', ('cur_cas',)),
                       ('sumeau',      ('ratio',)),
                       ('measles-usa', ('tot_cases',)),
                       ('ebolardc',    ('tot_confirmed',))):
    try:
        pf.setwhom(db, reload=False)
        available = set(pf.listwhich())
        for ind in indicators:
            want(f'{db}.which', ind, available)
    except Exception as exc:
        failures.append(f'setwhom({db!r}) raised {exc}')
        print(f'  FAIL setwhom({db!r}): {exc}')

print()
if failures:
    print(f'{len(failures)} problem(s):')
    for f in failures:
        print(' -', f)
else:
    print('all listings use a vocabulary the installed pyvoa recognises.')

## 2. Example 1 — comparing countries *(manuscript listing, Fig. 2)*

`where='European Union'` expands to the member states through `GeoRegion`;
`what='daily'` differentiates the cumulative series the source publishes;
`option='smooth7'` applies a weekly rolling mean, removing the reporting-day
artefact visible in every national series.

In [ ]:
import pyvoa.front as pf
pf.setwhom('owid')            # Our World in Data, worldwide, by country
pf.setvis('matplotlib')
pf.plot(which='total_deaths', where='European Union',
        what='daily', option='smooth7')

In [ ]:
save('fig2_timeseries_eu.png')

## 3. Example 2 — mapping a grouping *(manuscript listing, Fig. 3)*

One keyword selects the OECD member states, and the geolocation layer supplies
their geometries. The JHU series is read from its Zenodo mirror, the upstream
collection having ended in March 2023 — the point of the example as much as the
map itself.

`tile='positron'` rather than the default `'openstreet'`: OpenStreetMap's tile
usage policy blocks contextily, so the default draws the map over a grid of
"Access blocked" images.

In [ ]:
pf.setwhom('jhu')             # Johns Hopkins CSSE, worldwide
pf.map(which='tot_confirmed', where='OECD',
       what='daily', when='01/12/2022', tile='positron')

In [ ]:
save('fig3_map_oecd.png')

## 4. Example 3 — ranking, and leaving the framework *(manuscript listing, Fig. 4)*

`typeofhist='location'` draws one bar per country, ranked by magnitude rather
than by name. (`'value'` is a different chart: it bins the countries into a
frequency histogram.) The `get()` call returns the same selection as a pandas
frame, with one row per date and place and standardised geographic keys: pyvoa
is a starting point for an analysis, not an enclosure.

In [ ]:
pf.setwhom('owid')
pf.hist(which='total_people_vaccinated_per_hundred',
        where='Asia', typeofhist='location')

In [ ]:
save('fig4_hist_asia.png')

In [ ]:
df = pf.get(which='total_people_vaccinated_per_hundred',
            where='Asia', output='pandas')

print(type(df).__name__, df.shape)
print(list(df.columns))
df.head()

## 5. Example 4 — sub-national data *(manuscript listing, Fig. 5)*

The same four keywords address a sub-national French source at *département*
granularity — `typeofmap='dense'` bringing the overseas *départements* into the
frame rather than dropping them — and then a wastewater surveillance series,
whose yearly overlay exposes the seasonal structure that a chronological axis
hides. Neither call required knowing anything about the two providers' file
formats.

In [ ]:
pf.setwhom('spf')             # Sante publique France, departements
pf.map(which='cur_hosp', typeofmap='dense', tile='positron')

In [ ]:
save('fig5a_map_spf_dense.png')

In [ ]:
pf.setwhom('sumeau')          # SARS-CoV-2 in wastewater, France
pf.plot(which='ratio', typeofplot='yearly')

In [ ]:
save('fig5b_sumeau_yearly.png')

## 6. Example 5 — beyond COVID-19 *(manuscript listing, Fig. 6)*

The grammar does not change with the disease. The measles source publishes
county-level increments rather than a cumulative series: its descriptor says so,
and the counties of a state are summed and cumulated at parse time, so
`tot_cases` means here what it means for OWID. The Ebola source is indexed by
*zone de santé*, a geography the geolocation layer resolves — and whose provinces
it maps to their ISO 3166-2 codes — like French *départements* or US states.


In [ ]:
pf.setwhom('measles-usa')     # JHU measles tracking team, US states
pf.plot(which='tot_cases', where=['Texas', 'Utah', 'South Carolina'])


In [ ]:
save('fig6a_measles_usa.png')


In [ ]:
pf.setwhom('ebolardc')        # INSP situation reports, DRC health zones
pf.map(which='tot_confirmed', tile='positron')


In [ ]:
save('fig6b_ebola_drc.png')


## 7. The geolocation layer on its own *(not a manuscript listing)*

Supports the claim made in §2.1 and §4 that `pyvoa.geo` is importable
independently and contains nothing epidemiological. Any dataset keyed by
country, region or French *département* can be reconciled and mapped with it.

`GeoRegion.__init__` fetches about ten upstream pages, so the first run of this
cell is slower than the rest.

In [ ]:
import pyvoa.geo as pg

gm = pg.GeoManager('name')
print(gm.to_standard(['fr', 'US', 'china', "cote d'ivoire"], output='list'))

eu = gm.get_GeoRegion().get_countries_from_region('European Union')
print(len(eu), 'ISO3 codes, e.g.', eu[:5])

fra = pg.GeoCountry('FRA')
fra.get_region_list()

## 8. Wastewater against reported incidence *(not a manuscript listing — §4, point 1)*

The concrete illustration suggested for the Impact section: two unrelated
providers, one query grammar, one geographic key. Because both frames carry the
same `date` and `where` columns, putting the two signals on one axis is a merge
rather than a reconciliation exercise.

In [ ]:
pf.setwhom('sumeau')
waste = pf.get(which='ratio', what='current', output='pandas')
print('sumeau     ', waste.shape, waste['date'].min(), '..', waste['date'].max())

pf.setwhom('spfnational')
cases = pf.get(which='cur_cas', what='daily', option='smooth7',
               output='pandas')
print('spfnational', cases.shape, cases['date'].min(), '..', cases['date'].max())

merged = waste.merge(cases, on=['date', 'where'], suffixes=('_waste', '_cases'))
print('overlapping rows:', len(merged))
merged.head()

---

### Producing the manuscript figures

The `save()` cells above write PNG files, which is what the backends emit
directly. **Elsevier expects vector artwork**: re-export the retained figures as
PDF or EPS before submission, or set the matplotlib backend's DPI high enough
that the raster version meets the journal's minimum resolution.

A script version of this notebook, `paper_examples.py`, runs the same listings
headless (`matplotlib.use('Agg')`) and offers `--check` for the vocabulary test
of §1 alone. Either file, deposited alongside the manuscript or in
`examples/pyfiles/` of the repository, can serve as the reproducible capsule
requested in metadata fields C3 and S3.